# 04: Solver Validation — Convergence, Energy Conservation, and Ground Truth Certification

**Summary**: We present the complete validation package for our Split-Step Fourier Method
(SSFM) solver. The solver is validated against: (1) analytical solutions, (2) convergence
theory, and (3) conservation laws. The successful completion of all tests establishes the
solver as **trusted ground truth** for the Physics-Informed Neural Network (PINN) project.

**Validation results**:
- 2nd-order convergence verified (log-log slope = 2.0 +/- 0.3)
- Energy conserved near roundoff; validation pass threshold is 10⁻¹⁰
- Gaussian broadening matches analytical formula to < 2%
- Fundamental soliton propagates unchanged (max error < 10⁻⁴)

**Conclusion**: The solver passes all documented validation tests.

## Why Validation Matters

A numerical solver is only as useful as the trust we can place in its output. For our
purposes, this is critical: the solver will generate **training data for a physics-informed
neural network** (PINN-NLSE project). If the training data contains systematic errors,
the PINN will learn wrong physics.

We validate the solver at three levels:

| Level | What We Test | Method |
|-------|-------------|--------|
| **Correctness** | Does the solver reproduce known solutions? | Analytical soliton comparison |
| **Convergence** | Does refining the step size improve accuracy? | Log-log convergence plot |
| **Conservation** | Does the solver preserve the physics (energy)? | E(xi)/E(0) at every step |

In [1]:
import sys
from pathlib import Path
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise RuntimeError("Run from project root or notebooks/")

sys.path.insert(0, str(ROOT / "src"))

FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

from nlse_ssfm.validation import (
    run_convergence_study,
    run_energy_conservation_checks,
    run_gaussian_broadening_check,
    run_soliton_acid_test,
    run_spm_invariance_check,
    run_dispersion_spectral_power_check,
    run_higher_order_soliton_recurrence_checks,
)

## Test 1: Step-Size Convergence

We run the fundamental soliton ($N=1$) at six different step sizes and measure the maximum
amplitude error. The error should scale as $O(d\xi^2)$ — second-order convergence from
Strang (symmetric) splitting.

In [2]:
conv = run_convergence_study(save_path=FIG_DIR / "nb04_convergence.png")
slope = conv['slope']
complex_slope = conv['complex_slope']

print(f"Fitted convergence slope (shape): {slope:.2f}")
print(f"Fitted convergence slope (complex): {complex_slope:.2f}")
print(f"\nConvergence table (N_z, dxi, shape_error, complex_error):")
for row in conv['table']:
    print(f"  N_z={row[0]:5d}, dxi={row[1]:.6f}, "
          f"shape_err={row[2]:.3e}, complex_err={row[3]:.3e}")
assert abs(slope - 2.0) < 0.3
assert abs(complex_slope - 2.0) < 0.4

Fitted convergence slope (shape): 2.00
Fitted convergence slope (complex): 2.00

Convergence table (N_z, dxi, shape_error, complex_error):
  N_z=   50, dxi=0.157080, shape_err=1.213e-03, complex_err=1.256e-03
  N_z=  100, dxi=0.078540, shape_err=3.046e-04, complex_err=3.166e-04
  N_z=  200, dxi=0.039270, shape_err=7.636e-05, complex_err=7.936e-05
  N_z=  500, dxi=0.015708, shape_err=1.223e-05, complex_err=1.271e-05
  N_z= 1000, dxi=0.007854, shape_err=3.057e-06, complex_err=3.177e-06
  N_z= 2000, dxi=0.003927, shape_err=7.644e-07, complex_err=7.945e-07


The fitted shape-error slope confirms second-order convergence, consistent with
the Strang (symmetric) splitting. Halving the step size reduces the splitting error by 4x.

## Test 2: Energy Conservation

The NLSE conserves $E = \int |u|^2 d\tau$. The SSFM preserves this to floating-point
precision because each sub-step is **unitary**:
- Dispersion: FFT -> multiply by $e^{i\phi}$ ($|e^{i\phi}| = 1$) -> IFFT (Parseval)
- Nonlinear: multiply by $e^{iN^2|u|^2 d\xi}$ ($|e^{i\theta}| = 1$)

In [3]:
energy = run_energy_conservation_checks(
    save_path=FIG_DIR / "nb04_energy_conservation.png")
max_E_dev_N1 = energy['max_E_dev_N1']
max_E_dev_N2 = energy['max_E_dev_N2']
print(f"N=1 max energy deviation: {max_E_dev_N1:.2e}")
print(f"N=2 max energy deviation: {max_E_dev_N2:.2e}")
assert max_E_dev_N1 < 1e-10
assert max_E_dev_N2 < 1e-10

N=1 max energy deviation: 1.30e-13
N=2 max energy deviation: 3.16e-13


## Test 3: Analytical Comparison — Gaussian Broadening

For pure dispersion ($N^2 = 0$), the Gaussian pulse broadening has an exact formula:
$\sigma(\xi) / \sigma(0) = \sqrt{1 + \xi^2}$.

In [4]:
gaussian = run_gaussian_broadening_check(
    save_path=FIG_DIR / "nb04_analytical_comparison.png")
max_broadening_err = gaussian['width_error']
complex_gaussian_err = gaussian['complex_error']
print(f"Max width relative error: {max_broadening_err*100:.4f}%")
print(f"Final complex-field error: {complex_gaussian_err:.2e}")
assert max_broadening_err < 0.02
assert complex_gaussian_err < 1e-3

Max width relative error: 0.0000%
Final complex-field error: 1.35e-08


## Test 4: Fundamental Soliton — The Acid Test

The $N=1$ soliton $u(\xi,\tau) = \text{sech}(\tau) e^{i\xi/2}$ must propagate unchanged.
We require max shape error < 10⁻⁴ over 5 soliton periods.

In [5]:
soliton = run_soliton_acid_test(
    save_path=FIG_DIR / "nb04_soliton_acid_test.png")
max_soliton_err = soliton['max_shape_error']
exact_phase_err = soliton['exact_phase_error']
aligned_phase_err = soliton['aligned_phase_error']
print(f"Max N=1 soliton shape error: {max_soliton_err:.2e}")
print(f"Exact-phase complex error: {exact_phase_err:.2e}")
print(f"Phase-aligned complex error: {aligned_phase_err:.2e}")
assert max_soliton_err < 1e-4
assert exact_phase_err < 5e-3
assert aligned_phase_err < 1e-4

Max N=1 soliton shape error: 1.92e-05
Exact-phase complex error: 1.78e-04
Phase-aligned complex error: 1.27e-05


## Tests 5-7: Operator Limits and Higher-Order Recurrence

These checks verify pure SPM, pure dispersion, and N=2/N=3 recurrence. They catch
bugs hidden by the N=1 soliton test alone.

In [6]:
spm = run_spm_invariance_check()
max_spm_err = spm['max_intensity_deviation']
print(f"SPM-only max temporal-intensity error: {max_spm_err:.2e}")
assert max_spm_err < 1e-10

disp_spec = run_dispersion_spectral_power_check()
max_spec_err = disp_spec['spectral_power_error']
print(f"Dispersion-only spectral-power error: {max_spec_err:.2e}")
assert max_spec_err < 1e-10

higher = run_higher_order_soliton_recurrence_checks()
n2_rec = higher['N2_recurrence_half']
n3_rec = higher['N3_recurrence_half']
print(f"N=2 recurrence error at pi/2: {n2_rec:.2e}")
print(f"N=3 recurrence error at pi/2: {n3_rec:.2e}")
assert n2_rec < 5e-3
assert n3_rec < 1e-2

SPM-only max temporal-intensity error: 4.49e-13
Dispersion-only spectral-power error: 3.43e-13
N=2 recurrence error at pi/2: 1.39e-04
N=3 recurrence error at pi/2: 5.90e-04


## Validation Summary

All validation tests compiled in a single pass/fail table.

In [ ]:
def _status(condition):
    return "PASS" if condition else "FAIL"

results = [
    ("Shape convergence order",  "~2.0 +/- 0.3", f"{slope:.2f}",             abs(slope-2)<0.3),
    ("Complex convergence order","~2.0 +/- 0.4", f"{complex_slope:.2f}",      abs(complex_slope-2)<0.4),
    ("Energy (N=1)",             "< 1e-10",       f"{max_E_dev_N1:.1e}",       max_E_dev_N1<1e-10),
    ("Energy (N=2)",             "< 1e-10",       f"{max_E_dev_N2:.1e}",       max_E_dev_N2<1e-10),
    ("Gaussian broadening",      "< 2%",          f"{max_broadening_err*100:.2f}%", max_broadening_err<0.02),
    ("Gaussian complex field",   "< 1e-3",        f"{complex_gaussian_err:.1e}", complex_gaussian_err<1e-3),
    ("Soliton shape (N=1)",      "< 1e-4",        f"{max_soliton_err:.1e}",    max_soliton_err<1e-4),
    ("Soliton exact phase",      "< 5e-3",        f"{exact_phase_err:.1e}",    exact_phase_err<5e-3),
    ("Soliton aligned phase",    "< 1e-4",        f"{aligned_phase_err:.1e}",  aligned_phase_err<1e-4),
    ("SPM intensity invariant",  "< 1e-10",       f"{max_spm_err:.1e}",        max_spm_err<1e-10),
    ("Dispersion spectral power","< 1e-10",       f"{max_spec_err:.1e}",       max_spec_err<1e-10),
    ("N=2 recurrence",           "< 5e-3",        f"{n2_rec:.1e}",             n2_rec<5e-3),
    ("N=3 recurrence",           "< 1e-2",        f"{n3_rec:.1e}",             n3_rec<1e-2),
]

print("=" * 68)
print(f"{'Test':<30} {'Expected':<15} {'Measured':<15} {'Status'}")
print("=" * 68)
for name, expected, measured, passed in results:
    print(f"{name:<30} {expected:<15} {measured:<15} {_status(passed)}")
print("=" * 68)

all_pass = all(r[3] for r in results)
print(f"\n{'ALL TESTS PASSED' if all_pass else 'SOME TESTS FAILED'}")

Test                           Expected        Measured        Status
Shape convergence order        ~2.0 +/- 0.3    2.00            PASS
Complex convergence order      ~2.0 +/- 0.4    2.00            PASS
Energy (N=1)                   < 1e-10         1.3e-13         PASS
Energy (N=2)                   < 1e-10         3.2e-13         PASS
Gaussian broadening            < 2%            0.00%           PASS
Gaussian complex field         < 1e-3          1.3e-08         PASS
Soliton shape (N=1)            < 1e-4          1.9e-05         PASS
Soliton exact phase            < 5e-3          1.8e-04         PASS
Soliton aligned phase          < 1e-4          1.3e-05         PASS
SPM intensity invariant        < 1e-10         4.5e-13         PASS
Dispersion spectral power      < 1e-10         3.4e-13         PASS
N=2 recurrence                 < 5e-3          1.4e-04         PASS
N=3 recurrence                 < 1e-2          5.9e-04         PASS

ALL TESTS PASSED


: 

## Ground Truth Certification

Based on the validation results above, we certify that:

1. **The SSFM solver passes the validation suite** ? it reproduces exact analytical
   solutions to high accuracy
2. **The solver has known convergence order** ? 2nd-order (Strang splitting)
3. **The solver conserves the energy invariant** ? near roundoff, confirming unitary
   sub-steps
4. **The solver is ready for ground truth generation** ? trusted training data for
   companion PINN-NLSE experiments

### Recommended Parameters for Ground Truth Generation

| Scenario | N_t | N_z | Expected error |
|----------|-----|-----|---------------|
| Standard accuracy | 2048 | 500 | ~1e-5 |
| High accuracy | 2048 | 1000 | ~3e-6 |
| Reference accuracy | 4096 | 5000 | ~1e-7 |

### Reusable Solver Artifacts

| File | Contents |
|------|----------|
| `src/nlse_ssfm/ssfm.py` | Validated SSFM solver (3 functions) |
| `src/nlse_ssfm/nlse_utils.py` | Grid, pulse definitions, diagnostics |
| `src/nlse_ssfm/validation.py` | Importable solver-certification helpers |
| `figures/nb04_convergence.png` | Proof of 2nd-order convergence |
| `figures/nb04_energy_conservation.png` | Proof of energy preservation |